In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense
from tensorflow.keras.utils import to_categorical

In [ ]:
# Load text file
with open("data.txt", "r", encoding="utf-8") as file:
    text = file.read()

# Convert to lowercase
text = text.lower()

print("Sample text:")
print(text[:500])

Sample text:
hyderabad is a sprawling metropolis that stands as a majestic testament to india’s ability to blend ancient history with futuristic ambition. the city was founded in 1591 by muhammad quli qutb shah who envisioned a grand urban centre on the banks of the musi river. it originally served as the capital of the qutb shahi dynasty before becoming the seat of the immensely wealthy nizams. the iconic charminar stands at the heart of the old city and represents the architectural brilliance of the indo-i


In [ ]:
text

'hyderabad is a sprawling metropolis that stands as a majestic testament to india’s ability to blend ancient history with futuristic ambition. the city was founded in 1591 by muhammad quli qutb shah who envisioned a grand urban centre on the banks of the musi river. it originally served as the capital of the qutb shahi dynasty before becoming the seat of the immensely wealthy nizams. the iconic charminar stands at the heart of the old city and represents the architectural brilliance of the indo-islamic style. surrounding this monument are the bustling markets of laad bazaar where artisans have sold handcrafted bangles for centuries. the nearby mecca masjid is one of the oldest and largest mosques in india and reflects the deep spiritual heritage of the region. towering over the western edge of the city is the golconda fort which once guarded the world\'s most famous diamonds. this fortress is renowned for its ingenious acoustic systems that allowed messages to travel from the gates to 

In [ ]:
# Create tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])
tokenizer

In [ ]:
# Total number of unique words
total_words = len(tokenizer.word_index) + 1

print("Total Vocabulary Size:", total_words)

Total Vocabulary Size: 369


In [ ]:
#Create Training Sequences
input_sequences = []

for line in text.split("\n"):
    token_list = tokenizer.texts_to_sequences([line])[0]

    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

print("Total sequences:", len(input_sequences))

Total sequences: 627


In [ ]:
#Pad Sequences
max_sequence_len = max([len(seq) for seq in input_sequences])

input_sequences = pad_sequences(input_sequences,
                                maxlen=max_sequence_len,
                                padding='pre')

In [ ]:
max_sequence_len

628

In [ ]:
input_sequences

array([[  0,   0,   0, ...,   0,  10,   5],
       [  0,   0,   0, ...,  10,   5,   3],
       [  0,   0,   0, ...,   5,   3,  53],
       ...,
       [  0,   0,  10, ...,  13,   1, 366],
       [  0,  10,   5, ...,   1, 366, 367],
       [ 10,   5,   3, ..., 366, 367, 368]], dtype=int32)

In [ ]:
#Split Input and Output
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

# One hot encoding
y = to_categorical(y, num_classes=total_words)

In [ ]:
X

array([[  0,   0,   0, ...,   0,   0,  10],
       [  0,   0,   0, ...,   0,  10,   5],
       [  0,   0,   0, ...,  10,   5,   3],
       ...,
       [  0,   0,  10, ..., 365,  13,   1],
       [  0,  10,   5, ...,  13,   1, 366],
       [ 10,   5,   3, ...,   1, 366, 367]], dtype=int32)

In [ ]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 1., 0., 0.],
       [0., 0., 0., ..., 0., 1., 0.],
       [0., 0., 0., ..., 0., 0., 1.]])

In [ ]:
#Build RNN Model
rnn_model = Sequential()

rnn_model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))

rnn_model.add(SimpleRNN(150))

rnn_model.add(Dense(total_words, activation='softmax'))

rnn_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Build LSTM Model
lstm_model = Sequential()

lstm_model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))

lstm_model.add(LSTM(150))

lstm_model.add(Dense(total_words, activation='softmax'))

lstm_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
print("Training RNN Model...")
rnn_model.fit(X, y, epochs=50, verbose=1)

Training RNN Model...
Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 7s 245ms/step - accuracy: 0.0366 - loss: 5.8391
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 308ms/step - accuracy: 0.1200 - loss: 5.3221
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 244ms/step - accuracy: 0.1074 - loss: 5.2184
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 279ms/step - accuracy: 0.0857 - loss: 5.1852
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 263ms/step - accuracy: 0.1152 - loss: 5.0847
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 245ms/step - accuracy: 0.1295 - loss: 4.9319
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 6s 309ms/step - accuracy: 0.1325 - loss: 4.8751
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 293ms/step - accuracy: 0.1621 - loss: 4.5978
Epoch 9/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 5s 253ms/step - accuracy: 0.1902 - loss: 4.4360
Epoch 10/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 311ms/step - accuracy: 0.2018 - loss: 4.1144
Epoch 11/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 308ms/step - accuracy: 0.2330 - loss: 3.9005
Epoch 12/50
20/20 ━━━━━━━━━

In [ ]:
print("Training LSTM Model...")
lstm_model.fit(X, y, epochs=50, verbose=1)

Training LSTM Model...
Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 23s 1s/step - accuracy: 0.0681 - loss: 5.8893
Epoch 2/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 20s 968ms/step - accuracy: 0.1229 - loss: 5.3317
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.1081 - loss: 5.2919
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 959ms/step - accuracy: 0.1001 - loss: 5.2946
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 22s 1s/step - accuracy: 0.1177 - loss: 5.1616
Epoch 6/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 970ms/step - accuracy: 0.0867 - loss: 5.3249
Epoch 7/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.0972 - loss: 5.1819
Epoch 8/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 970ms/step - accuracy: 0.1039 - loss: 5.0398
Epoch 9/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 962ms/step - accuracy: 0.1108 - loss: 5.0593
Epoch 10/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 21s 1s/step - accuracy: 0.1255 - loss: 4.8807
Epoch 11/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 962ms/step - accuracy: 0.1456 - loss: 4.6507
Epoch 12/50
20/20 ━━━━━━━━━━━━━━━

In [ ]:
def generate_text(seed_text, next_words, model, temperature=1.0):

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list],
                                   maxlen=max_sequence_len-1,
                                   padding='pre')

        preds = model.predict(token_list, verbose=0)[0]

        preds = np.log(preds + 1e-7) / temperature
        preds = np.exp(preds) / np.sum(np.exp(preds))

        predicted_index = np.random.choice(len(preds), p=preds)

        predicted_word = tokenizer.index_word.get(predicted_index, "")

        seed_text += " " + predicted_word

    return seed_text

In [ ]:
print("\nGenerated Text using RNN:")
print(generate_text("Ramoji Film City", 10, rnn_model))


Generated Text using RNN:
Ramoji Film City is a politics to the ramoji film city which is


In [ ]:
print("\nGenerated Text using LSTM:")

print(generate_text("Ramoji Film City", 10, lstm_model))


Generated Text using LSTM:
Ramoji Film City is a sprawling sprawling metropolis metropolis stands as a majestic
